In [1]:
%%capture
!pip install textacy
!pip install faiss-gpu
!pip install chromadb
!pip install nltk
!pip install sentence_transformers

In [2]:
%%capture
# Initiliase Ollama server on a thread

#Download ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread
#Download model
!ollama pull llama3
!pip install ollama
import ollama
model_name = 'llama3'

Couldn't find '/home/jovyan/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAINRDxmMf3QX8bo2nwV5UcTib6PiULT+GUQ9TJXeKQRnv



2025/04/05 02:47:17 routes.go:1231: INFO server config env="map[CUDA_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:2048 OLLAMA_DEBUG:false OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_INTEL_GPU:false OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/home/jovyan/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_SCHED_SPREAD:false ROCR_VISIBLE_DEVICES: http_proxy: https

[GIN] 2025/04/05 - 02:47:18 | 200 |      93.803µs |       127.0.0.1 | HEAD     "/"


time=2025-04-05T02:47:18.509Z level=INFO source=download.go:177 msg="downloading 6a0746a1ec1a in 16 291 MB part(s)"
time=2025-04-05T02:47:59.668Z level=INFO source=download.go:177 msg="downloading 4fa551d4f938 in 1 12 KB part(s)"
time=2025-04-05T02:48:00.815Z level=INFO source=download.go:177 msg="downloading 8ab4849b038c in 1 254 B part(s)"
time=2025-04-05T02:48:01.979Z level=INFO source=download.go:177 msg="downloading 577073ffcc6c in 1 110 B part(s)"
time=2025-04-05T02:48:03.133Z level=INFO source=download.go:177 msg="downloading 3f8eb4da87fa in 1 485 B part(s)"


[GIN] 2025/04/05 - 02:48:16 | 200 | 58.011550462s |       127.0.0.1 | POST     "/api/pull"


In [3]:
import re
import asyncio
import pandas as pd
import numpy as np
from pydantic import BaseModel
from ollama import AsyncClient
import textacy.datasets
import requests
import json
from sentence_transformers import SentenceTransformer
from textwrap import shorten
import nltk
from textacy.representations.vectorizers import Vectorizer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import torch

In [4]:
# Load Supreme Court dataset
sc = textacy.datasets.SupremeCourt()
sc.download()

# Area code classification for 15 labels
# print(sc.issue_area_codes)

mapper = sc.issue_area_codes

# Create a DataFrame from the dataset
cases = [{"id": record.meta["issue"] ,"text": record.text , "actual_class": str(mapper[record.meta['issue_area']]) } for record in sc.records()]

cases_df = pd.DataFrame(cases)

#cases_df = cases_df[:10]
# Display the first few rows of the dataset
cases_df.head()
# cases_df = cases_df[:100]

100%|██████████| 111M/111M [00:01<00:00, 62.9MB/s] 


,id,text,actual_class
0,80180,[ Halliburton Oil Well Cementing Co. v. Walker...,Economic Activity
1,10500,"Rehearing Denied Dec. 16, 1946. See . Mr.Claud...",Criminal Procedure
2,80250,"Rehearing Denied Dec. 16, 1946\n See .\n Appea...",Economic Activity
3,20150,"Mr.\nWalter J. Cummings, Jr., of Washington, D...",Civil Rights
4,80060,"Mr.A. Devitt Vaneck, of Washington, D.C., for ...",Economic Activity


In [5]:
# Get words from model's perpective
response = ollama.chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': '''Give me a list of words and phrases (atleast 15) which would be present in a case text to classify it into following categories. Do not write any explaination or warnings.
            - Criminal Procedure: Cases involving the process of investigating and prosecuting crimes.
            - Civil Rights: Cases about the rights of individuals to receive equal treatment.
            - First Amendment: Cases addressing issues related to freedom of speech, religion, or press.
            - Due Process: Cases focusing on legal safeguards ensuring fair treatment.
            - Privacy: Cases dealing with the right to privacy.
            - Attorneys: Cases about the legal profession and lawyer regulations.
            - Unions: Cases concerning labor unions and collective bargaining.
            - Economic Activity: Cases involving business, trade, or commerce.
            - Judicial Power: Cases addressing the powers of courts and the judiciary.
            - Federalism: Cases about the division of power between state and federal governments.
            - Interstate Relations: Cases about interactions between states.
            - Federal Taxation: Cases involving federal tax laws.
            - Miscellaneous: Cases that do not clearly fit into any of the above categories.
            - Private Action: Cases involving disputes between private individuals or entities.

    Format the output in the following way:
    'Category': words, phrase ...
      ''',
  },
])

time=2025-04-05T02:48:31.400Z level=WARN source=ggml.go:149 msg="key not found" key=llama.vision.block_count default=0
time=2025-04-05T02:48:31.400Z level=WARN source=ggml.go:149 msg="key not found" key=llama.attention.key_length default=128
time=2025-04-05T02:48:31.400Z level=WARN source=ggml.go:149 msg="key not found" key=llama.attention.value_length default=128
time=2025-04-05T02:48:31.400Z level=INFO source=sched.go:716 msg="new model will fit in available VRAM in single GPU, loading" model=/home/jovyan/.ollama/models/blobs/sha256-6a0746a1ec1aef3e7ec53868f220ff6e389f6f8ef87a01d77c96807de94ca2aa gpu=GPU-c60ea6aa-b04c-e2b1-2508-cb11d68e68e9 parallel=4 available=8409448448 required="6.2 GiB"
time=2025-04-05T02:48:31.612Z level=INFO source=server.go:105 msg="system memory" total="251.4 GiB" free="246.3 GiB" free_swap="2.0 GiB"
time=2025-04-05T02:48:31.612Z level=WARN source=ggml.go:149 msg="key not found" key=llama.vision.block_count default=0
time=2025-04-05T02:48:31.613Z level=WARN s

[GIN] 2025/04/05 - 02:48:50 | 200 | 19.079584571s |       127.0.0.1 | POST     "/api/chat"


In [6]:
# print(response['message']['content'])

def parse_text_to_dict(text):
    result_dict = {}
    lines = text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line or ':' not in line:
            continue  # Ignore headers or tails
        match = re.match(r"^(.*?):\s*(.*)$", line)
        if match:
            category = match.group(1).strip()
            values = [word.strip() for word in match.group(2).split(',')]
            result_dict[category] = values
    
    return result_dict

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)  # Ensure input is a string
    return re.sub(r'[^a-zA-Z0-9 ]', '', text).strip().lower()

def filter_result_dict(result_dict, valid_categories):
    cleaned_valid_categories = {clean_text(str(value)) for value in valid_categories.values()}
    cleaned_result_dict = {clean_text(key): values for key, values in result_dict.items()}
    return {key: values for key, values in cleaned_result_dict.items() if key in cleaned_valid_categories}

# Convert text to dictionary
result = parse_text_to_dict(response['message']['content'])
# for i in result:
#     print(i)
#     print(result[i])

filtered_result = filter_result_dict(result, sc.issue_area_codes)
for i in filtered_result:
    print(i)
    print(filtered_result[i])
    

criminal procedure
['arrest warrant', 'search and seizure', 'Miranda rights', 'preliminary hearing', 'grand jury', 'indictment', 'trial', 'evidence', 'procedure', 'investigation ...']
civil rights
['equal protection', 'discrimination', 'segregation', 'affirmative action', 'voting rights', 'race', 'gender', 'sexual orientation', 'disability', 'national origin ...']
first amendment
['freedom of speech', 'freedom of the press', 'religion', 'expression', 'assembly', 'petition', 'censorship', 'libel', 'slander', 'prior restraint ...']
due process
['notice', 'hearing', 'judgment', 'review', 'appeal', 'procedural due process', 'substantive due process', 'Fourteenth Amendment', 'Fifth Amendment ...']
privacy
['privacy rights', 'surveillance', 'wiretap', 'search and seizure', 'data protection', 'confidentiality', 'intrusion', 'trespass', 'invasion of privacy ...']
attorneys
['lawyer-client privilege', 'attorney-client relationship', 'ethics', 'malpractice', 'fees', 'disciplinary action', 'bar a

In [7]:
# Convert dataset records to a DataFrame
records = list(sc.records())
df = pd.DataFrame(records)

# Display the first few rows
print(df.head())

                                                text  \
0  [ Halliburton Oil Well Cementing Co. v. Walker...   
1  Rehearing Denied Dec. 16, 1946. See . Mr.Claud...   
2  Rehearing Denied Dec. 16, 1946\n See .\n Appea...   
3  Mr.\nWalter J. Cummings, Jr., of Washington, D...   
4  Mr.A. Devitt Vaneck, of Washington, D.C., for ...   

                                                meta  
0  {'issue': '80180', 'issue_area': 8, 'n_min_vot...  
1  {'issue': '10500', 'issue_area': 1, 'n_min_vot...  
2  {'issue': '80250', 'issue_area': 8, 'n_min_vot...  
3  {'issue': '20150', 'issue_area': 2, 'n_min_vot...  
4  {'issue': '80060', 'issue_area': 8, 'n_min_vot...  


In [ ]:
# Download necessary resources
nltk.download("stopwords")

# def classify_case_rag(case_text, classification_dict):
#     """
#     Dynamically classify a case using Retrieval-Augmented Generation (RAG) by querying relevant legal topics
#     from all categories in the dictionary.
    
#     Args:
#         case_text (str): The text of the case to be classified.
#         classification_dict (dict): Dictionary mapping classification categories to relevant keywords.
    
#     Returns:
#         str: The best-matching class name for the case.
#     """
    
#     # Load the embedding model
#     model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    
#     # Define the ChromaDB path
#     chroma_db_path = "./chroma_db"
    
#     # Initialize a persistent ChromaDB client
#     chroma_client = chromadb.PersistentClient(path=chroma_db_path)
    
#     # Check if the collection already exists
#     existing_collections = chroma_client.list_collections()
#     if "temp_case_classification" in [col for col in existing_collections]:
#         collection = chroma_client.get_collection("temp_case_classification")
#         print(0)
#     else:
#         collection = chroma_client.create_collection(name="temp_case_classification")

#         # Populate ChromaDB dynamically with classification categories and keywords
#         for class_name, keywords in classification_dict.items():
#             for keyword in keywords:
#                 embedding = model.encode(keyword).tolist()
#                 collection.add(embeddings=[embedding], metadatas=[{"class_name": class_name}], ids=[keyword])
    
#     # Encode the case text for retrieval
#     case_embedding = model.encode(case_text).tolist()
    
#     # Retrieve the most relevant legal categories
#     results = collection.query(query_embeddings=[case_embedding], n_results=3)
    
#     retrieved_classes = [
#         res["class_name"] for res in results["metadatas"][0]
#     ] if results["metadatas"] else []
    
#     if not retrieved_classes:
#         return "No relevant classification found"
    
#     # Prepare relevant context for LLaMA 3
#     retrieved_text = "\n".join(retrieved_classes)

#     prompt = f"""
#     You are an AI assistant trained in Supreme Court case classification. Given the following retrieved categories,
#     classify the new case into the most appropriate issue area.

#     Retrieved Relevant Categories:
#     {retrieved_text}



#     Please reply with the category only and nothing else."""

#     #  Case Text:
#     # {case_text[:1000]}  # Limiting case text for concise processing
    
#     # Query LLaMA 3 using the Ollama instance
#     response = ollama.chat(model='llama3', messages=[{"role": "user", "content": prompt}])
#     # Explicitly delete case_embedding to free memory
#     del case_embedding
#     torch.cuda.empty_cache()  # Clear unused memory from GPU
    
#     return response['message']['content']


import torch
from sentence_transformers import SentenceTransformer

def classify_case_rag(case_text, classification_dict):
    """
    Dynamically classify a case using Retrieval-Augmented Generation (RAG) by querying relevant legal topics.
    
    Args:
        case_text (str): The text of the case to be classified.
        classification_dict (dict): Dictionary mapping classification categories to relevant keywords.
    
    Returns:
        str: The best-matching class name for the case.
    """
    
    # Check GPU availability
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load the embedding model and move it to GPU
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)

    # If multiple GPUs, use DataParallel
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for inference.")
        model = torch.nn.DataParallel(model)
    
    # Function to handle encoding with multi-GPU support
    def encode_text(text):
        if torch.cuda.device_count() > 1:
            return model.module.encode(text, convert_to_tensor=True).to(device)
        else:
            return model.encode(text, convert_to_tensor=True).to(device)
    
    # Define the ChromaDB path
    chroma_db_path = "./chroma_db"
    
    # Initialize a persistent ChromaDB client
    chroma_client = chromadb.PersistentClient(path=chroma_db_path)
    
    # Check if the collection already exists
    existing_collections = chroma_client.list_collections()
    if "temp_case_classification" in [col.name for col in existing_collections]:
        collection = chroma_client.get_collection("temp_case_classification")
    else:
        collection = chroma_client.create_collection(name="temp_case_classification")

        # Populate ChromaDB dynamically with classification categories and keywords
        for class_name, keywords in classification_dict.items():
            for keyword in keywords:
                embedding = encode_text(keyword)
                collection.add(embeddings=[embedding.cpu().tolist()], 
                               metadatas=[{"class_name": class_name}], 
                               ids=[keyword])
    
    # Encode the case text for retrieval (ensure it's on GPU)
    case_embedding = encode_text(case_text)

    # Retrieve the most relevant legal categories
    results = collection.query(query_embeddings=[case_embedding.cpu().tolist()], n_results=3)
    
    retrieved_classes = [
        res["class_name"] for res in results["metadatas"][0]
    ] if results["metadatas"] else []
    
    if not retrieved_classes:
        return "No relevant classification found"
    
    # Prepare relevant context for LLaMA 3
    retrieved_text = "\n".join(retrieved_classes)

    prompt = f"""
    You are an AI assistant trained in Supreme Court case classification. Given the following retrieved categories,
    classify the new case into the most appropriate issue area.

    Retrieved Relevant Categories:
    {retrieved_text}

    Please reply with the category only and nothing else. Ensure its category is from the given list or the most relevant one. Here are the categories:
            - None: Cases that do not fit into any specific category.
            - Criminal Procedure: Cases involving the process of investigating and prosecuting crimes.
            - Civil Rights: Cases about the rights of individuals to receive equal treatment.
            - First Amendment: Cases addressing issues related to freedom of speech, religion, or press.
            - Due Process: Cases focusing on legal safeguards ensuring fair treatment.
            - Privacy: Cases dealing with the right to privacy.
            - Attorneys: Cases about the legal profession and lawyer regulations.
            - Unions: Cases concerning labor unions and collective bargaining.
            - Economic Activity: Cases involving business, trade, or commerce.
            - Judicial Power: Cases addressing the powers of courts and the judiciary.
            - Federalism: Cases about the division of power between state and federal governments.
            - Interstate Relations: Cases about interactions between states.
            - Federal Taxation: Cases involving federal tax laws.
            - Miscellaneous: Cases that do not clearly fit into any of the above categories.
            - Private Action: Cases involving disputes between private individuals or entities.
    """

    # Query LLaMA 3 using the Ollama instance
    response = ollama.chat(model='llama3', messages=[{
                "role": "system",
                "content": '''
                You are an expert in Supreme Court cases and legal classifications. Your task is to analyze the provided text of a Supreme Court case and classify it into one of the following predefined categories:
                
                Instructions:
                1. Carefully analyze the text and identify the primary category it belongs to.
                2. Provide only the category name as the response and nothing else.
                3. If uncertain, select the most relevant category based on the case's primary legal focus.
                4. Response should not exceed more than 2 words.
                ''',
            },{"role": "user", "content": prompt}])
    
    # Free GPU memory
    del case_embedding
    torch.cuda.empty_cache()

    return response['message']['content']


[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# %%capture
# Process and classify cases
import gc
classified_cases = []

def process_cases():
    """
    Process all cases synchronously and classify them using the local API.
    """
    for idx, row in cases_df.iterrows():
        print(f"Processing case {idx + 1}/{len(cases_df)}...")

        # Classify the case
        with torch.no_grad():
            class_name = classify_case_rag(row["text"], filtered_result)  # Pass the preprocessed text

        if class_name:
            classified_cases.append({"id": row["id"], "predicted_class_name": class_name})
        else:
            print(f"Failed to classify case {idx + 1}")
        torch.cuda.empty_cache()
        gc.collect()

        # break
    
    print("Processing completed.")

# Run the case processing
process_cases()



Processing case 1/8419...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:48:57 | 200 |  303.530105ms |       127.0.0.1 | POST     "/api/chat"
Processing case 2/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:48:58 | 200 |  344.593335ms |       127.0.0.1 | POST     "/api/chat"
Processing case 3/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:49:00 | 200 |  240.281603ms |       127.0.0.1 | POST     "/api/chat"
Processing case 4/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:49:01 | 200 |  203.624804ms |       127.0.0.1 | POST     "/api/chat"
Processing case 5/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:49:02 | 200 |  199.297476ms |       127.0.0.1 | POST     "/api/chat"
Processing case 6/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:49:03 | 200 |  229.436982ms |       127.0.0.1 | POST     "/api/chat"
Processing case 7/8419...
Using 2 GPUs for inference.
[GIN] 2025/04/05 - 02:49:04 | 200 |  233.988243ms |       127.0.0.1 | POST     "/api/chat"
Processing 

In [ ]:
# Save the results to a DataFrame
classified_df = pd.DataFrame(classified_cases)

# filling none for Nan values
classified_df.fillna({'class_name': 'None'}, inplace=True)

# Display the results
classified_df.head()

In [ ]:
merged_df = pd.merge(cases_df,classified_df, left_index=True,right_index=True)  # Use "left", "right", or "outer" if needed
merged_df.head()

In [ ]:
merged_df.to_csv("classified_rag_llama3.csv")

In [ ]:
from sklearn.metrics import accuracy_score

# Assuming your DataFrame is named df
accuracy = accuracy_score(merged_df["actual_class"], merged_df["predicted_class_name"])
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(merged_df["actual_class"], merged_df["predicted_class_name"], labels=merged_df["actual_class"].unique())

# Create a heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=merged_df["actual_class"].unique(), yticklabels=merged_df["actual_class"].unique())
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import time

# Assuming merged_df contains your actual and predicted classes
# Create a new directory based on the current timestamp
timestamp = time.strftime("%Y%m%d-%H%M%S")
folder_name = f"run_{timestamp}"
os.makedirs(folder_name, exist_ok=True)

# Save the CSV file
csv_file_path = os.path.join(folder_name, "classified_rag_llama3.csv")
merged_df.to_csv(csv_file_path, index=False)

# Calculate accuracy
accuracy = accuracy_score(merged_df["actual_class"], merged_df["predicted_class_name"])
print(f"Accuracy: {accuracy:.2%}")

# Save accuracy to a text file
accuracy_file_path = os.path.join(folder_name, "accuracy.txt")
with open(accuracy_file_path, "w") as f:
    f.write(f"Accuracy: {accuracy:.2%}\n")

# Compute confusion matrix
cm = confusion_matrix(merged_df["actual_class"], merged_df["predicted_class_name"], labels=merged_df["actual_class"].unique())

# Create a heatmap of the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=merged_df["actual_class"].unique(), yticklabels=merged_df["actual_class"].unique())
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Confusion Matrix")

# Save the confusion matrix plot as a PNG file
confusion_matrix_png_path = os.path.join(folder_name, "confusion_matrix.png")
plt.savefig(confusion_matrix_png_path)

# Close the plot to free up memory
plt.close()

print(f"Files saved in folder: {folder_name}")
